# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [ ]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

# EDA

In [ ]:
# Check rating range
print("\n--- Full Ratings Dataset ---")
print(f"Min rating: {ratings_df['rating'].min()}")
print(f"Max rating: {ratings_df['rating'].max()}")

# Unique keywords: some rows have empty lists, which is not null but still missing data
unique_keywords = keywords_df['id'].nunique()
print(f"\nUnique keywords: {unique_keywords}")

In [ ]:

print("Credits: \n", credits_df.isnull().sum())
print("\nKeywords: \n", keywords_df.isnull().sum())
print("\nLinks: \n", links_df.isnull().sum())
print("\nRatings: \n", ratings_df .isnull().sum())
print("\nMetadata: \n", movies_metadata_df.isnull().sum())

In [ ]:
# Sample of movies_metadata_df null values in title
print("\nLinks: \n")
print(movies_metadata_df.loc[
    movies_metadata_df["title"].isnull(),
    ["original_title", "title"],
])

In [ ]:
bad_title = movies_metadata_df[movies_metadata_df["title"].isnull()]
print(bad_title[["id", "adult", "original_title", "release_date", "vote_count"]])

ids_numeric = pd.to_numeric(movies_metadata_df["id"], errors="coerce")
print("Non-numeric ids:", movies_metadata_df.loc[ids_numeric.isnull(), "id"].tolist())

# 'adult' should only be True/False
print(movies_metadata_df["adult"].value_counts().head())

In [ ]:
null_title_idx = movies_metadata_df.index[movies_metadata_df["title"].isnull()]
is_tail = pd.to_numeric(movies_metadata_df.loc[null_title_idx, "id"], errors="coerce").isnull()
tail_idx = null_title_idx[is_tail]
head_idx = null_title_idx[~is_tail]

assert len(head_idx) == len(tail_idx) == 3
assert (head_idx + 1 == tail_idx).all()

recovered_dates = movies_metadata_df.loc[tail_idx, "id"].values
assert pd.to_datetime(recovered_dates, format="%Y-%m-%d", errors="coerce").notnull().all()
movies_metadata_df.loc[head_idx, "release_date"] = recovered_dates

movies_metadata_df.loc[head_idx, "title"] = movies_metadata_df.loc[head_idx, "original_title"]

# Drop the malformed tail rows
movies_metadata_df = movies_metadata_df.drop(index=tail_idx)

# Verify
print("Null titles remaining:", movies_metadata_df["title"].isnull().sum())
print("Non-numeric ids remaining:", pd.to_numeric(movies_metadata_df["id"], errors="coerce").isnull().sum())
print(movies_metadata_df.loc[head_idx, ["id", "original_title", "title", "release_date"]])

In [ ]:
# movies_metadata sample of ids
print("\nSample of movies_metadata_df ids:")
print(movies_metadata_df.loc[head_idx, ["id", "imdb_id"]])

print("\nClean imdb_id in movies_metadata_df")
# Check how many start with 'tt'
sample_imdb = movies_metadata_df['imdb_id'].dropna().astype(str)
starts_with_tt = sample_imdb.str.startswith('tt').all()
print(f"All non-null imdb_id start with 'tt': {starts_with_tt}")

# Remove 'tt' prefix and convert remainder to nullable integer
movies_metadata_df['imdb_id_clean'] = movies_metadata_df['imdb_id'].astype(str).str.replace('tt', '', regex=False)
movies_metadata_df['imdb_id_clean'] = pd.to_numeric(movies_metadata_df['imdb_id_clean'], errors='coerce').astype('Int64')

print("Sample cleaned imdb_id values:")
print(movies_metadata_df[['imdb_id', 'imdb_id_clean']].head(3))

# Drop the old imdb_id column and rename the clean version
movies_metadata_df = movies_metadata_df.drop(columns=['imdb_id'])
movies_metadata_df = movies_metadata_df.rename(columns={'imdb_id_clean': 'imdb_id'})
